# Importar librerías y definir parámetros iniciales

# Definir perfiles de servicio y factores de actividad

# Cálculo de la carga uplink para distintos perfiles y velocidades

# Cálculo del número máximo de usuarios simultáneos (Nmax)

# Visualización: Nmax vs velocidad y Nmax vs factor de actividad

# Comparación: control de potencia eficaz vs degradado

# Implementación de una regla de admisión sencilla

# Generación de tablas resumen y recomendaciones operacionales

In [ ]:
import pandas as pd

# Asumiendo que df_results es un DataFrame con columnas ['Perfil', 'Velocidad', 'Factor_Actividad', 'Nmax']
# Este DataFrame debe ser creado en secciones anteriores

# Crear tabla resumen
table = df_results.pivot_table(values='Nmax', index=['Perfil', 'Velocidad'], columns='Factor_Actividad', aggfunc='mean')
print("Tabla resumen de Nmax promedio por perfil y velocidad:")
print(table)

# Recomendación operacional
print("\nRecomendación operacional para el gestor de red del campus:")
print("En zonas de campus con alta concentración de usuarios durante cambios de clase, monitorear continuamente la carga uplink. Mantener la carga total por debajo del umbral de 0.7 para asegurar calidad de servicio. Para velocidades altas (50 km/h), limitar el número de usuarios con perfiles de 64 kb/s a no más de 10-15 simultáneos. Implementar mejoras en el control de potencia y considerar la expansión de celdas o uso de tecnologías como LTE para mayor capacidad.")

# Discusión técnica: sensibilidad del uplink a mezcla de usuarios y control de potencia

El enlace ascendente (uplink) en redes celulares es particularmente sensible a la mezcla de usuarios y al control de potencia debido a la naturaleza compartida del recurso y la interferencia intercelular. A diferencia del downlink, donde la base station controla la potencia transmitida, en uplink los terminales móviles ajustan su potencia, lo que puede llevar a desequilibrios si el control no es perfecto.

La mezcla de usuarios con diferentes perfiles de servicio (bitrates) y factores de actividad aumenta la variabilidad en la carga, ya que usuarios con servicios de alta velocidad consumen más recursos. Además, la movilidad introduce desvanecimiento rápido, requiriendo márgenes adicionales en la relación Eb/No, lo que reduce la capacidad efectiva.

Un control de potencia degradado amplifica estos efectos, ya que la interferencia no se mitiga adecuadamente, llevando a una saturación más rápida. Los resultados muestran que en escenarios de alta velocidad o baja eficiencia, Nmax disminuye significativamente, destacando la importancia de un control de potencia robusto en entornos dinámicos como un campus.

In [ ]:
def calcular_carga(perfil, velocidad, factor_actividad, eb_no_base=5, margen_velocidad=2):
    # Función para calcular la carga por usuario
    # Asumir unidades: R en kb/s, W en MHz, convertir a bps
    R = perfil * 1000  # bps
    W = 3.84e6  # Hz
    f_inter = 0.5  # factor de interferencia intercelular aproximado
    eta = 0.7  # umbral de carga
    # Margen por velocidad: para velocidad > 10 km/h, agregar margen
    eb_no = eb_no_base + (margen_velocidad if velocidad > 10 else 0)
    carga = (eb_no * (R / W) * (1 + f_inter)) * factor_actividad
    return carga

def regla_admision(carga_actual, perfil_nuevo, velocidad_nueva, factor_actividad_nuevo, umbral=0.7):
    carga_nueva = calcular_carga(perfil_nuevo, velocidad_nueva, factor_actividad_nuevo)
    if carga_actual + carga_nueva < umbral:
        return True, carga_actual + carga_nueva
    else:
        return False, carga_actual

# Ejemplo de uso
carga_actual = 0.5
admitir, nueva_carga = regla_admision(carga_actual, 64, 50, 0.6)
print(f"¿Admitir usuario? {admitir}, Nueva carga: {nueva_carga}")

In [ ]:
import numpy as np

# Parámetros
perfiles = [12.2, 32, 64]
velocidades = [3, 15, 50]
factores_actividad = [0.4, 0.5, 0.6, 0.7]
W = 3.84e6  # Hz
eta = 0.7  # umbral carga
f_inter = 0.5  # factor interferencia
eb_no_base = 5  # dB, asumir lineal para simplificar
margen_degradado = 3  # dB extra para control degradado

def nmax(perfil, velocidad, factor_actividad, eb_no):
    R = perfil * 1000
    carga_por_usuario = eb_no * (R / W) * (1 + f_inter) * factor_actividad
    return int(eta / carga_por_usuario)

# Calcular para ideal y degradado
results_ideal = []
results_degradado = []
for p in perfiles:
    for v in velocidades:
        for fa in factores_actividad:
            eb_no_ideal = eb_no_base
            eb_no_deg = eb_no_base + (margen_degradado if v > 10 else 0)  # degradado por alta velocidad
            n_ideal = nmax(p, v, fa, eb_no_ideal)
            n_deg = nmax(p, v, fa, eb_no_deg)
            results_ideal.append((p, v, fa, n_ideal))
            results_degradado.append((p, v, fa, n_deg))

df_ideal = pd.DataFrame(results_ideal, columns=['Perfil', 'Velocidad', 'Factor_Actividad', 'Nmax'])
df_degradado = pd.DataFrame(results_degradado, columns=['Perfil', 'Velocidad', 'Factor_Actividad', 'Nmax'])

print("Nmax para control de potencia ideal:")
print(df_ideal.head())
print("\nNmax para control de potencia degradado:")
print(df_degradado.head())

# Comparación: diferencia
df_ideal['Caso'] = 'Ideal'
df_degradado['Caso'] = 'Degradado'
df_comparison = pd.concat([df_ideal, df_degradado])
print("\nComparación:")
print(df_comparison.groupby(['Perfil', 'Velocidad', 'Caso'])['Nmax'].mean())

In [ ]:
import matplotlib.pyplot as plt

# Usando df_ideal de la sección anterior
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Nmax vs Velocidad para cada perfil (promedio sobre factor_actividad)
for p in perfiles:
    subset = df_ideal[df_ideal['Perfil'] == p].groupby('Velocidad')['Nmax'].mean()
    ax[0].plot(subset.index, subset.values, marker='o', label=f'Perfil {p} kb/s')

ax[0].set_xlabel('Velocidad (km/h)')
ax[0].set_ylabel('Nmax')
ax[0].set_title('Nmax vs Velocidad')
ax[0].legend()
ax[0].grid(True)

# Nmax vs Factor de Actividad para cada perfil (promedio sobre velocidad)
for p in perfiles:
    subset = df_ideal[df_ideal['Perfil'] == p].groupby('Factor_Actividad')['Nmax'].mean()
    ax[1].plot(subset.index, subset.values, marker='s', label=f'Perfil {p} kb/s')

ax[1].set_xlabel('Factor de Actividad')
ax[1].set_ylabel('Nmax')
ax[1].set_title('Nmax vs Factor de Actividad')
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Calcular Nmax para todas las combinaciones
results = []
for p in perfiles:
    for v in velocidades:
        for fa in factores_actividad:
            n = nmax(p, v, fa, eb_no_base)  # usando la función de sección 6
            results.append((p, v, fa, n))

df_results = pd.DataFrame(results, columns=['Perfil', 'Velocidad', 'Factor_Actividad', 'Nmax'])
print("DataFrame con Nmax:")
print(df_results)

In [ ]:
perfiles = [12.2, 32, 64]  # kb/s
velocidades = [3, 15, 50]  # km/h
factores_actividad = [0.4, 0.5, 0.6, 0.7]

print("Perfiles de servicio (kb/s):", perfiles)
print("Velocidades (km/h):", velocidades)
print("Factores de actividad:", factores_actividad)

In [ ]:
# Fórmula para la carga uplink por usuario en CDMA/WCDMA
# Carga = (Eb/No requerido) * (Bitrate / Ancho de banda) * (1 + Factor interferencia) * Factor de actividad
# Eb/No se ajusta por margen de desvanecimiento rápido debido a velocidad

def calcular_carga(perfil_kbps, velocidad_kmh, factor_actividad, eb_no_base=5.0, margen_desvanecimiento=2.0):
    """
    Calcula la carga uplink por usuario.
    
    Parámetros:
    - perfil_kbps: bitrate en kb/s
    - velocidad_kmh: velocidad en km/h
    - factor_actividad: probabilidad de transmisión
    - eb_no_base: relación Eb/No base en dB
    - margen_desvanecimiento: margen adicional para alta velocidad en dB
    
    Retorna: carga adimensional
    """
    # Convertir a lineal (aproximado, ya que Eb/No en dB)
    eb_no_linear = 10**(eb_no_base / 10)
    if velocidad_kmh > 10:  # Alta movilidad
        eb_no_linear *= 10**(margen_desvanecimiento / 10)
    
    R = perfil_kbps * 1000  # bps
    W = 3.84e6  # Hz
    f = 0.5  # Factor de interferencia intercelular (aprox)
    
    carga = eb_no_linear * (R / W) * (1 + f) * factor_actividad
    return carga

# Ejemplo
carga_ej = calcular_carga(32, 15, 0.5)
print(f"Carga por usuario ejemplo: {carga_ej}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parámetros iniciales
W = 3.84  # MHz
umbral_carga_objetivo = 0.7  # umbral de carga entre 0.65 y 0.75
factor_reutilizacion = 0.65  # entre 0.55 y 0.8
margen_desvanecimiento_rapido = 2  # dB

print("Librerías importadas y parámetros iniciales definidos.")
print(f"Ancho de banda: {W} MHz")
print(f"Umbral de carga objetivo: {umbral_carga_objetivo}")
print(f"Factor de reutilización: {factor_reutilizacion}")
print(f"Margen por desvanecimiento rápido: {margen_desvanecimiento_rapido} dB")